# 04_Fingerprint Machine Learning

## Objective
The objective of this notebook is to train and evaluate fingerprint recognition models.

## Method
Machine learning training and evaluation workflow.

## Input
Fingerprint feature dataset.

## Processing Tasks

1. Load and Inspect Fingerprint Dataset.
2. Finger Type Identification.
3. Prepare Features and Labels.
4. Configure GroupKFold Cross-Validation.
5. Configure Feature Scaling.
6. Train Random Forest Model.
7. Train Support Vector Machine (SVM) Model.
8. Train Final SVM Model.
9. Save Final SVM Model.

## Output
- Fingerprint model metrics.
- Performance comparison results.
- Saved best fingerprint model.

## Purpose
Select a robust fingerprint model for multimodal verification integration.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Project Paths
PROJECT_PATH = Path(r"E:\Multimodal Biometric Verification System")

FEATURES_PATH = (
    PROJECT_PATH
    / "Dataset"
    / "Processed"
    / "Fingerprint"
    / "Fingerprint_Features"
    / "Fingerprint_Combined_Features.csv"
)

MODEL_PATH = PROJECT_PATH / "Models" / "Fingerprint"

# Configuration
RANDOM_STATE = 42
TEST_SIZE = 0.20

USERS = ["User_01", "User_02"]

# Create Model Directory
MODEL_PATH.mkdir(parents=True, exist_ok=True)

print("Fingerprint Machine Learning Environment Configured")
print("---------------------------------------------")
print(f"Features Path: {FEATURES_PATH}")
print(f"Model Path: {MODEL_PATH}")
print(f"Selected Users: {USERS}")
print(f"Random State: {RANDOM_STATE}")
print(f"Test Size: {TEST_SIZE}")

### 1. Load and Inspect Fingerprint Dataset


In [ ]:
# Load Dataset
fingerprint_df = pd.read_csv(FEATURES_PATH)

# Basic Dataset Information
print("Fingerprint Dataset Loaded Successfully")
print("---------------------------------------------")
print(f"Dataset Shape: {fingerprint_df.shape}")

# User Distribution
print("\nUser Distribution:")
print(fingerprint_df["User_ID"].value_counts())

# Selected Users Validation
print("\nSelected Users:")
print(sorted(fingerprint_df["User_ID"].unique()))

# Missing Values
print("\nMissing Values:")
print(fingerprint_df.isnull().sum().sum())

# Feature Information
metadata_columns = ["User_ID", "Image_Name"]
feature_columns = [
    column for column in fingerprint_df.columns
    if column not in metadata_columns
]

print("\nFeature Information:")
print(f"Metadata Columns: {len(metadata_columns)}")
print(f"Numerical Features: {len(feature_columns)}")

# Duplicate Records
print(f"\nDuplicate Records: {fingerprint_df.duplicated().sum()}")

# Dataset Preview
print("\nDataset Preview:")
display(fingerprint_df.head())

### 2. Finger Type Identification

This section extracts the finger type from each image filename.

Images belonging to the same finger type across different users will be assigned to the same group.

These groups will be used later with GroupKFold Cross-Validation to prevent the same finger type from appearing in both training and testing sets within a fold.


In [ ]:
# Extract Finger Type from Image Names
fingerprint_df["Finger_Type"] = fingerprint_df.apply(
    lambda row: row["Image_Name"]
    .replace(f"{row['User_ID']}_", "")
    .rsplit(".", 1)[0],
    axis=1
)

# Display Finger Type Distribution
print("Finger Type Identification Completed")
print("---------------------------------------------")

print("\nFinger Type Distribution:")
print(fingerprint_df["Finger_Type"].value_counts().sort_index())

# Verify Number of Users per Finger Type
print("\nUsers per Finger Type:")
finger_groups = fingerprint_df.groupby("Finger_Type")["User_ID"].nunique()
print(finger_groups)

# Validate Group Structure
print("\nGroup Validation:")
print(f"Total Finger Types: {fingerprint_df['Finger_Type'].nunique()}")
print(f"Each Finger Type Has Both Users: {finger_groups.eq(2).all()}")

# Check One Image per User for Each Finger Type
images_per_group = fingerprint_df.groupby(
    ["Finger_Type", "User_ID"]
).size()

print(
    f"One Image per User per Finger Type: "
    f"{images_per_group.eq(1).all()}"
)

### 3. Prepare Features and Labels

The Finger_Type column will be retained separately for group-based cross-validation, while User_ID will be used as the classification target.


In [ ]:
# Define Feature Columns
feature_columns = [
    column for column in fingerprint_df.columns
    if column not in ["User_ID", "Image_Name", "Finger_Type"]
]

# Prepare Features and Labels
X = fingerprint_df[feature_columns].copy()
y = fingerprint_df["User_ID"].copy()

# Prepare Groups
groups = fingerprint_df["Finger_Type"].copy()

# Display Dataset Information
print("Fingerprint Features and Labels Prepared")
print("---------------------------------------------")
print(f"Features Shape: {X.shape}")
print(f"Labels Shape: {y.shape}")
print(f"Groups Shape: {groups.shape}")

print(f"\nNumber of Numerical Features: {len(feature_columns)}")
print(f"Number of Finger Types: {groups.nunique()}")

print("\nUser Distribution:")
print(y.value_counts())

print("\nFeature Data Type:")
print(X.dtypes.value_counts())

### 4. Configure GroupKFold Cross-Validation


In [ ]:
# Configure GroupKFold
from sklearn.model_selection import GroupKFold

N_SPLITS = 5

group_kfold = GroupKFold(n_splits=N_SPLITS)

print("GroupKFold Configuration")
print("---------------------------------------------")
print(f"Number of Splits: {N_SPLITS}")

# Inspect Fold Distribution
for fold, (train_idx, test_idx) in enumerate(
    group_kfold.split(X, y, groups=groups), start=1
):
    train_groups = groups.iloc[train_idx].unique()
    test_groups = groups.iloc[test_idx].unique()

    print(f"\nFold {fold}:")
    print(f"Training Images: {len(train_idx)}")
    print(f"Testing Images: {len(test_idx)}")
    print(f"Training Finger Types: {len(train_groups)}")
    print(f"Testing Finger Types: {len(test_groups)}")
    print(f"Training Users: {y.iloc[train_idx].value_counts().to_dict()}")
    print(f"Testing Users: {y.iloc[test_idx].value_counts().to_dict()}")
    print(f"Group Overlap: {len(set(train_groups) & set(test_groups))}")

### 5. Configure Feature Scaling


StandardScaler standardizes numerical features by removing the mean and scaling them to unit variance.

Scaling will be performed inside an sklearn Pipeline during cross-validation to prevent data leakage. Random Forest will use the original features without scaling.


In [ ]:
# Configure SVM Pipeline
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf",
        C=1,
        gamma="scale"
    ))
])

print("SVM Pipeline Configured Successfully")
print("---------------------------------------------")
print("Scaling: StandardScaler")
print("Classifier: Support Vector Machine")
print("Kernel: RBF")
print("C: 1")
print("Gamma: scale")

print("\nRandom Forest:")
print("Scaling: Not Required")

### 6. Train Random Forest Model

The model will be trained and evaluated using the original HOG and LBP features without feature scaling.


In [ ]:
# Configure Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# Configure Evaluation Metrics
scoring = {
    "accuracy": "accuracy",
    "precision": "precision_weighted",
    "recall": "recall_weighted",
    "f1": "f1_weighted"
}

# Evaluate Random Forest with GroupKFold
rf_cv_results = cross_validate(
    rf_model,
    X,
    y,
    cv=group_kfold.split(X, y, groups=groups),
    scoring=scoring,
    n_jobs=-1,
    return_train_score=True
)

# Display Results
print("Random Forest Cross-Validation Results")
print("---------------------------------------------")

for metric in scoring:
    scores = rf_cv_results[f"test_{metric}"]
    print(f"\n{metric.upper()}:")
    print(f"Fold Scores: {np.round(scores, 4)}")
    print(f"Mean Score: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")

print("\nTraining Accuracy:")
print(np.round(rf_cv_results["train_accuracy"], 4))
print(f"Mean Training Accuracy: {rf_cv_results['train_accuracy'].mean():.4f}")

### 7. Train Support Vector Machine (SVM) Model


In [ ]:
# Evaluate SVM with GroupKFold
svm_cv_results = cross_validate(
    svm_pipeline,
    X,
    y,
    cv=group_kfold.split(X, y, groups=groups),
    scoring=scoring,
    n_jobs=-1,
    return_train_score=True
)

# Display Results
print("SVM Cross-Validation Results")
print("---------------------------------------------")

for metric in scoring:
    scores = svm_cv_results[f"test_{metric}"]
    print(f"\n{metric.upper()}:")
    print(f"Fold Scores: {np.round(scores, 4)}")
    print(f"Mean Score: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")

print("\nTraining Accuracy:")
print(np.round(svm_cv_results["train_accuracy"], 4))
print(f"Mean Training Accuracy: {svm_cv_results['train_accuracy'].mean():.4f}")

### 8. Train Final SVM Model


In [ ]:
# Train Final SVM Model on the Complete Dataset
final_svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf",
        C=1,
        gamma="scale"
    ))
])

final_svm_model.fit(X, y)

print("Final SVM Model Trained Successfully")
print("---------------------------------------------")
print(f"Training Samples: {len(X)}")
print(f"Number of Features: {X.shape[1]}")
print(f"Selected Users: {sorted(y.unique())}")
print("Classifier: SVM")
print("Kernel: RBF")
print("Scaling: StandardScaler")

### 9. Save Final SVM Model

This section saves the trained SVM Pipeline and its feature column names.

The saved bundle includes the trained StandardScaler, SVM classifier, and feature column order to ensure consistent feature processing during runtime inference.


In [ ]:
# Save Final SVM Model Bundle
MODEL_PATH.mkdir(parents=True, exist_ok=True)

fingerprint_model_bundle = {
    "model": final_svm_model,
    "feature_columns": feature_columns
}

FINGERPRINT_MODEL_FILE = MODEL_PATH / "SVM_Fingerprint_Recognition.joblib"

joblib.dump(
    fingerprint_model_bundle,
    FINGERPRINT_MODEL_FILE
)

print("Fingerprint SVM Model Saved Successfully")
print("---------------------------------------------")
print(f"Model File: {FINGERPRINT_MODEL_FILE}")
print(f"Feature Columns: {len(feature_columns)}")
print(f"Training Samples: {len(X)}")
print(f"Selected Users: {sorted(y.unique())}")
print(f"File Exists: {FINGERPRINT_MODEL_FILE.exists()}")

In [ ]:
# Load Saved Model Bundle
loaded_fingerprint_bundle = joblib.load(FINGERPRINT_MODEL_FILE)

loaded_fingerprint_model = loaded_fingerprint_bundle["model"]
loaded_feature_columns = loaded_fingerprint_bundle["feature_columns"]

# Generate Predictions
original_predictions = final_svm_model.predict(X[feature_columns])
loaded_predictions = loaded_fingerprint_model.predict(X[loaded_feature_columns])

# Verify Model
print("Fingerprint SVM Model Verification")
print("---------------------------------------------")

print(f"Model Loaded: {loaded_fingerprint_model is not None}")
print(f"Feature Columns Match: {feature_columns == loaded_feature_columns}")
print(f"Number of Features: {len(loaded_feature_columns)}")

print(
    f"Predictions Match: "
    f"{np.array_equal(original_predictions, loaded_predictions)}"
)

print(f"Loaded Model Predictions: {loaded_predictions}")